# SpaceX Falcon 9 First Stage Landing Prediction
## Lab 6: Interactive Visual Analytics with Folium

We place every launch site on an interactive map, add color-coded markers for successful and failed landings, and measure distances from a launch site to its nearby coastline, highway, railway and city.

In [1]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

In [2]:
spacex_df = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv")
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


### Task 1: Mark all launch sites on a map

In [3]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=4)

circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                 html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC'))
site_map.add_child(circle)
site_map.add_child(marker)

for _, site in launch_sites_df.iterrows():
    coordinate = [site['Lat'], site['Long']]
    folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(site['Launch Site'])).add_to(site_map)
    folium.map.Marker(
        coordinate,
        icon=DivIcon(icon_size=(20, 20), icon_anchor=(0, 0),
                     html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site['Launch Site'])).add_to(site_map)

site_map.save("images/map_launch_sites.html")
site_map

All launch sites are close to the coast, and they sit as far south as possible in the continental US (Florida and California), close to the Equator.

### Task 2: Mark the success/failed launches for each site on the map

In [4]:
marker_cluster = MarkerCluster()
spacex_df['marker_color'] = spacex_df['class'].apply(lambda c: 'green' if c == 1 else 'red')
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [5]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
site_map.add_child(marker_cluster)

for _, record in spacex_df.iterrows():
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=record['Launch Site'])
    marker_cluster.add_child(marker)

site_map.save("images/map_launch_outcomes.html")
site_map

### Task 3: Calculate the distances between a launch site to its proximities

In [6]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)
site_map.add_child(mouse_position)

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

In [7]:
launch_site_lat, launch_site_lon = 28.563197, -80.576820

proximities = {
    'Coastline': (28.56319, -80.56785),
    'Highway (Samuel C Phillips Pkwy)': (28.56335, -80.57085),
    'Railway (NASA Railroad)': (28.57206, -80.58528),
    'City (Cape Canaveral)': (28.40199, -80.60440),
}

distances = {name: calculate_distance(launch_site_lat, launch_site_lon, lat, lon) for name, (lat, lon) in proximities.items()}
pd.DataFrame({'Proximity': list(distances.keys()), 'Distance (km)': [round(d, 2) for d in distances.values()]})

,Proximity,Distance (km)
0,Coastline,0.88
1,Highway (Samuel C Phillips Pkwy),0.58
2,Railway (NASA Railroad),1.29
3,City (Cape Canaveral),18.13


In [8]:
proximity_map = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=13)
folium.Circle([launch_site_lat, launch_site_lon], radius=100, color='#000000', fill=True).add_child(folium.Popup('CCAFS SLC-40')).add_to(proximity_map)
folium.map.Marker(
    [launch_site_lat, launch_site_lon],
    icon=DivIcon(icon_size=(120, 20), icon_anchor=(0, 0),
                 html='<div style="font-size: 13px; color:#000000;"><b>CCAFS SLC-40</b></div>')).add_to(proximity_map)

colors = ['#1f77b4', '#ff7f0e', '#9467bd', '#d62728']
anchors = [(0, 0), (60, 28), (0, 0), (0, 0)]
for (name, (lat, lon)), color, anchor in zip(proximities.items(), colors, anchors):
    distance = distances[name]
    folium.Marker(
        [lat, lon],
        icon=DivIcon(icon_size=(160, 20), icon_anchor=anchor,
                     html='<div style="font-size: 13px; color:%s;"><b>%s: %.2f KM</b></div>' % (color, name.split(' (')[0], distance))).add_to(proximity_map)
    folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [lat, lon]], weight=3, color=color).add_to(proximity_map)

proximity_map.add_child(MousePosition(position='topright', separator=' Long: ', prefix='Lat:'))
proximity_map.save("images/map_proximities.html")
proximity_map

* Launch sites are close to railways and highways, which makes transporting rockets and equipment easier.
* Launch sites are very close to the coastline, so launches fly over the ocean and failed stages fall away from people.
* Launch sites keep a safe distance from cities (around 18 km to Cape Canaveral).